# L6n2 inductor: Palace simulation against measured data

Simulate the exact `L_6n2` layout from Volker Mühlhaus's
[measured-versus-simulated benchmark](https://github.com/VolkerMuehlhaus/gds2palace_ihp_sg13g2/tree/101e119fc08469fa3547eface93e1edc2dddfad7/more_examples/measured_vs_simulated/more_accurate_models_L6n2_v2)
using gsim. Compare the two-port S-parameters, differential inductance, resistance,
Q, and self-resonant frequency with the supplied de-embedded measurement.
The benchmark's name is L6n2; its measured differential inductance near 1 GHz is about 5 nH.

Use the current gsim 0.6+ project's `.venv` kernel; the existing `palace-inductor` dependency group
supplies `ihp-gdsfactory`. Run from the repository root or `nbs/`.
The first run downloads checksum-verified GDS, stackups, and Touchstone files from
a pinned upstream commit. Downloads retain their upstream license and stay in the
ignored `palace-sim-l6n2/assets/` cache. No gds2palace implementation is imported.

**Default behavior:** build and validate the mesh, report tetrahedra and estimated
field DOFs, and plot the measured and archived upstream results. Set `RUN_SOLVER=True`
to run a fresh gsim simulation on the cloud or a local Palace installation.
Archived curves are explicitly labeled; they are not results from this notebook.

In [ ]:
import hashlib
import json
import urllib.request
import xml.etree.ElementTree as ET
from importlib.metadata import version
from pathlib import Path

import gdsfactory as gf
import gmsh
import klayout.db as kdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skrf as rf
from ihp import PDK
from IPython.display import display as show_table

import gsim
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.palace import DrivenSim

PDK.activate()
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/gsim").is_dir()
)
work_dir = project_root / "nbs/palace-sim-l6n2"
asset_dir = work_dir / "assets"
asset_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
STACK_MODEL = "conformal"  # 'conformal' or 'planar'
MESH_SIZE_UM = 2.0  # 5: initial run; 2: validation; 1: convergence check
FEM_ORDER = 2
MESH_ALGORITHM_3D = "delaunay"  # reproducible current gsim default
BUILD_MESH = True
RUN_SOLVER = False
SOLVER_BACKEND = "cloud"  # 'cloud' requires GF+ credentials; or 'local'
LOCAL_PROCESSES = 4
FMIN_HZ, FMAX_HZ = 0.1e9, 14e9
NUM_POINTS = 140  # 100 MHz spacing; no DC solve
ADAPTIVE_TOL = 1e-3
DIELECTRIC_MARGIN_UM = 100.0
AIR_MARGIN_UM = 50.0
SHOW_MESH = False  # enable the interactive 3D viewer
SAVED_GSIM_RESULTS = []  # optional paths to previous gsim .s2p files for convergence comparisons

assert STACK_MODEL in {"conformal", "planar"}
assert SOLVER_BACKEND in {"cloud", "local"}
assert MESH_SIZE_UM > 0
assert FEM_ORDER in {1, 2, 3, 4}
run_dir = work_dir / f"{STACK_MODEL}-{MESH_SIZE_UM:g}um-order{FEM_ORDER}"

## Exact benchmark inputs

The GDS includes TopMetal1, TopMetal2, 128 TopVia2 cuts, the artificial common
ground on layer 250, and port markers on 201/202. Import that layout instead of
generating a similar spiral with a PDK PCell. The measured file has two 50 Ω ports
and covers 0.1–50 GHz; comparison below uses the simulated 0.1–14 GHz band.

Upstream files are from commit `101e119fc08469fa3547eface93e1edc2dddfad7`.
The source repository declares GPL-3.0-or-later; its license is downloaded alongside
the data. The public files do not establish an OSS tapeout provenance.

In [ ]:
upstream_commit = "101e119fc08469fa3547eface93e1edc2dddfad7"
upstream_root = (
    "https://raw.githubusercontent.com/VolkerMuehlhaus/"
    f"gds2palace_ihp_sg13g2/{upstream_commit}"
)
benchmark_path = "more_examples/measured_vs_simulated/more_accurate_models_L6n2_v2"
checksums = {
    "L6n2_with_ports.gds": "30dd46f38e30a7cf1677076be2a0b64049bdadbe9da82d7235891481555f3ddd",
    "meas_L5_6n2_THRU_deemb.S2P": "cbfb4cfcd777ca9f958de4f1db7d43caeee2ab18dc9523f8b9350fa3ecc274cd",
    "SG13G2_200um.xml": "2615d4d75d6d81e0314b42ecdedf754b9e856d826bf829cb7e14081d8fb1b044",
    "SG13G2_200um_conformal.xml": "9c9bb2c3e462d2b2a2d3f880916570b3148155495ea9235c84d5119f72ae3702",
    "results/L6n2_with_ports_1um_passi3D.s2p": "3f949f5ccbfe17a36b6d86c524103855881ffa75bfd49cfe4aa3374de2b00bb3",
    "results/L6n2_with_ports_2um_passi3D.s2p": "7f5ed1cbdf818a0f74dcc894d3fecd97adfd8e9abd376557aecbb962d6df9384",
    "results/L6n2_with_ports_5um_passi3D.s2p": "e861173eeff49ed9ed3bb164bf35bc6ca49ec5b347774f04562da89d9d260bed",
    "results/L6n2_with_ports_2um_mergecorrection_filled.s2p": "ba256359bfc489c4bc08c199ce368825cbf5ae7c1e953c93e8b3162dbae30aa9",
}


def fetch_asset(relative_path, expected_hash):
    destination = asset_dir / relative_path
    if not destination.exists():
        with urllib.request.urlopen(  # noqa: S310 -- fixed HTTPS source, verified SHA-256
            f"{upstream_root}/{benchmark_path}/{relative_path}", timeout=60
        ) as response:
            contents = response.read()
        if hashlib.sha256(contents).hexdigest() != expected_hash:
            raise ValueError(f"Upstream checksum mismatch: {relative_path}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(contents)
    if hashlib.sha256(destination.read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Cached checksum mismatch: {destination}")
    return destination


assets = {name: fetch_asset(name, digest) for name, digest in checksums.items()}
license_path = asset_dir / "LICENSE.upstream"
if not license_path.exists():
    with urllib.request.urlopen(f"{upstream_root}/LICENSE", timeout=60) as response:  # noqa: S310 -- fixed HTTPS source
        license_path.write_bytes(response.read())
print(f"Verified {len(assets)} benchmark files from {upstream_commit[:12]}")

## Measurement and published simulation references

For equal and opposite terminal currents, use
$Z_\mathrm{diff}=Z_{11}-Z_{12}-Z_{21}+Z_{22}$,
$L=\operatorname{Im}(Z_\mathrm{diff})/(2\pi f)$,
$R=\operatorname{Re}(Z_\mathrm{diff})$, and
$Q=\operatorname{Im}(Z_\mathrm{diff})/\operatorname{Re}(Z_\mathrm{diff})$.
This is the total differential inductance, with no additional factor of two.
Define SRF as the first positive-to-negative reactance crossing after 1 GHz.
Peak Q is measured within the inductive region of the comparison band.

In [ ]:
measured = rf.Network(str(assets["meas_L5_6n2_THRU_deemb.S2P"]))
archived = {
    f"Upstream conformal {size} µm (archived)": rf.Network(
        str(assets[f"results/L6n2_with_ports_{size}um_passi3D.s2p"])
    )
    for size in (5, 2, 1)
}
archived["Upstream planar 2 µm (archived)"] = rf.Network(
    str(assets["results/L6n2_with_ports_2um_mergecorrection_filled.s2p"])
)


def differential_data(network):
    mask = (network.f >= FMIN_HZ) & (network.f <= FMAX_HZ)
    frequency = network.f[mask]
    z = network.z[mask]
    impedance = z[:, 0, 0] - z[:, 0, 1] - z[:, 1, 0] + z[:, 1, 1]
    quality = np.divide(
        impedance.imag,
        impedance.real,
        out=np.full(len(impedance), np.nan),
        where=impedance.real > 0,
    )
    return pd.DataFrame(
        {
            "f_GHz": frequency / 1e9,
            "L_nH": impedance.imag / (2 * np.pi * frequency) * 1e9,
            "R_ohm": impedance.real,
            "Q": quality,
            "X_ohm": impedance.imag,
        }
    )


def metrics(network):
    data = differential_data(network)
    x = data.X_ohm.to_numpy()
    crossings = np.flatnonzero((x[:-1] > 0) & (x[1:] <= 0))
    crossings = [index for index in crossings if data.f_GHz.iloc[index] >= 1]
    srf = np.nan
    if crossings:
        index = crossings[0]
        srf = np.interp(
            0, [-x[index], -x[index + 1]], data.f_GHz.iloc[index : index + 2]
        )
    inductive = data[
        (data.X_ohm > 0) & (data.f_GHz < srf if np.isfinite(srf) else True)
    ]
    valid_quality = inductive.dropna(subset=["Q"])
    peak = valid_quality.loc[valid_quality.Q.idxmax()] if len(valid_quality) else None
    return {
        "L @ 1 GHz (nH)": np.interp(1, data.f_GHz, data.L_nH),
        "R @ 1 GHz (Ω)": np.interp(1, data.f_GHz, data.R_ohm),
        "Peak Q": peak.Q if peak is not None else np.nan,
        "f at peak Q (GHz)": peak.f_GHz if peak is not None else np.nan,
        "SRF (GHz)": srf,
    }


reference_networks = {"Measured (de-embedded)": measured, **archived}
reference_metrics = pd.DataFrame(
    {name: metrics(net) for name, net in reference_networks.items()}
).T
show_table(reference_metrics.round(3))


def s_parameter_errors(network, reference):
    mask = (network.f >= max(FMIN_HZ, reference.f[0])) & (
        network.f <= min(FMAX_HZ, reference.f[-1])
    )
    frequency = network.f[mask]
    if not len(frequency):
        raise ValueError("No common frequencies for S-parameter comparison")
    measured_s = np.empty((len(frequency), 2, 2), dtype=complex)
    for row in range(2):
        for column in range(2):
            values = reference.s[:, row, column]
            measured_s[:, row, column] = np.interp(
                frequency, reference.f, values.real
            ) + 1j * np.interp(frequency, reference.f, values.imag)
    error = np.abs(network.s[mask] - measured_s)
    return {"RMS |ΔS|": np.sqrt(np.mean(error**2)), "Max |ΔS|": error.max()}

## Geometry and via correction

Keep the imported metal polygons. Merge each 4×4 via array into a block and multiply
its conductivity by the original-area / merged-area ratio to preserve its DC conductance.
Disable gsim's automatic via merging so it cannot also merge the filled spiral geometry.
Derive the 0.6 µm oxide surround on 900/901 as specified in the conformal XML.
Layer 902 defines the background dielectric footprint, extending 100 µm beyond the GDS.

Place two 8 µm wide, Z-directed 50 Ω port sheets at the source marker x positions,
from the common ground's top surface at z=0 to the bottom of TopMetal1 at z=6.4303 µm.
Align the source markers with the actual lead edges (a ≤5 nm adjustment in y)
to avoid nanometer-wide fragments at the terminal boundaries.

In [ ]:
source_layout = kdb.Layout()
source_layout.read(str(assets["L6n2_with_ports.gds"]))
source_cell = source_layout.cell("L_6n2")
port_definitions = []
source_leads = kdb.Region(
    source_cell.begin_shapes_rec(source_layout.layer(126, 0))
).merged()
for name, marker in [("P1", 201), ("P2", 202)]:
    bounds = source_cell.bbox(source_layout.layer(marker, 0)).to_dtype(
        source_layout.dbu
    )
    lead_bounds = [
        polygon.bbox().to_dtype(source_layout.dbu) for polygon in source_leads.each()
    ]
    terminal_edge = min(
        box.bottom for box in lead_bounds if box.left <= bounds.center().x <= box.right
    )
    port_definitions.append((name, (bounds.center().x, terminal_edge), bounds.width()))

original = gf.import_gds(assets["L6n2_with_ports.gds"], cellname="L_6n2").copy()
component = original.copy()
component.flatten()
assert np.isclose(component.kcl.dbu, 0.001), (
    "gsim polygon extraction expects a 1 nm DBU"
)


def region_on(part, gds_layer):
    layer_index = part.kcl.layout.layer(*gds_layer)
    return kdb.Region(part.kdb_cell.begin_shapes_rec(layer_index)).merged()


via_cuts = region_on(component, (133, 0))
assert via_cuts.count() == 128
merge_radius = round(1.0 / component.kcl.dbu)
via_blocks = via_cuts.sized(merge_radius).merged().sized(-merge_radius).merged()
assert via_blocks.count() == 8
fill_factors = np.array(
    [
        (via_cuts & kdb.Region(block)).area() / block.area()
        for block in via_blocks.each()
    ]
)
assert np.allclose(fill_factors, fill_factors[0])
via_fill_factor = float(fill_factors[0])
via_layer_index = component.kcl.layout.layer(133, 0)
component.kdb_cell.shapes(via_layer_index).clear()
component.kdb_cell.shapes(via_layer_index).insert(via_blocks)

if STACK_MODEL == "conformal":
    metal2 = region_on(component, (134, 0))
    surround = metal2.sized(round(0.6 / component.kcl.dbu)).merged()
    component.kdb_cell.shapes(component.kcl.layout.layer(900, 0)).insert(surround)
    component.kdb_cell.shapes(component.kcl.layout.layer(901, 0)).insert(
        surround - metal2
    )

chip_bounds = original.dbbox()
dielectric_bounds = chip_bounds.enlarged(DIELECTRIC_MARGIN_UM)
component.add_polygon(
    [
        (dielectric_bounds.left, dielectric_bounds.bottom),
        (dielectric_bounds.right, dielectric_bounds.bottom),
        (dielectric_bounds.right, dielectric_bounds.top),
        (dielectric_bounds.left, dielectric_bounds.top),
    ],
    layer=(902, 0),
)
for name, center, width in port_definitions:
    component.add_port(
        name=name,
        center=center,
        width=width,
        orientation=270,
        layer=(126, 0),
        port_type="electrical",
    )
print(
    f"Via fill factor: {via_fill_factor:.6f}; effective conductivity: {3.143e6 * via_fill_factor:.0f} S/m"
)
show_table(
    pd.DataFrame(port_definitions, columns=["Port", "Center (µm)", "Width (µm)"])
)

In [ ]:
from matplotlib.patches import Polygon as PlotPolygon

fig, ax = plt.subplots(figsize=(6, 7))
for layer, color, label in [
    ((250, 0), "lightgray", "Common ground"),
    ((126, 0), "steelblue", "TopMetal1"),
    ((134, 0), "darkorange", "TopMetal2"),
    ((133, 0), "black", "TopVia2 blocks"),
]:
    for index, polygon in enumerate(region_on(component, layer).each()):
        vertices = [
            (point.x * component.kcl.dbu, point.y * component.kcl.dbu)
            for point in polygon.each_point_hull()
        ]
        ax.add_patch(
            PlotPolygon(
                vertices, color=color, alpha=0.7, label=label if index == 0 else None
            )
        )
for name, center, width in port_definitions:
    ax.plot([center[0] - width / 2, center[0] + width / 2], [center[1]] * 2, "r-", lw=3)
    ax.annotate(name, center, xytext=(0, -15), textcoords="offset points", ha="center")
ax.autoscale_view()
ax.set(
    aspect="equal",
    xlabel="x (µm)",
    ylabel="y (µm)",
    title="Exact imported L6n2 geometry",
)
ax.legend(loc="upper right", fontsize=8)
plt.show()

## Benchmark stack and filled metals

Read material constants and metal heights from the supplied planar XML, rather than
the generic PDK defaults. The substrate is 180 µm of silicon (εr=11.9, σ=2 S/m),
with a 3.75 µm epitaxial layer (σ=5 S/m). TopMetal1 is 2 µm thick and TopMetal2 is 3 µm.

For `conformal`, use 12.7303 µm background SiO₂, a 0.4 µm blanket passive layer,
and the XML's derived SiO₂ volumes: 0.6 µm lateral coverage and 1.9 µm above TopMetal2.
For `planar`, use the original 15.7303 µm blanket SiO₂ plus 0.4 µm passive layer.
gsim's conductive-volume (`via`) layer role retains filled TopMetal1/TopMetal2 volumes
and their finite conductivities; it does not replace them with PEC or impedance shells.
The artificial ground is modeled as a PEC sheet at its top reference plane. Upstream
uses a finite-thickness nearly PEC ground block, so this is a small modeling difference.

In [ ]:
xml_root = ET.parse(assets["SG13G2_200um.xml"]).getroot()  # noqa: S314 -- pinned, checksum-verified XML
materials = {}
for entry in xml_root.findall("Materials/Material"):
    props = {"type": entry.get("Type").lower()}
    for source_key, target_key in [
        ("Conductivity", "conductivity"),
        ("Permittivity", "permittivity"),
        ("DielectricLossTangent", "loss_tangent"),
    ]:
        if source_key in entry.attrib:
            props[target_key] = float(entry.get(source_key))
    materials[entry.get("Name")] = props
materials["TopVia2"]["conductivity"] *= via_fill_factor
layers = {}


def add_layer(name, gds_layer, zmin, zmax, material, role):
    layers[name] = Layer(
        name=name,
        gds_layer=gds_layer,
        zmin=zmin,
        zmax=zmax,
        thickness=zmax - zmin,
        material=material,
        layer_type=role,
    )


for entry in xml_root.findall("ELayers/Layers/Layer"):
    if entry.get("Name") in {"TopMetal1", "TopMetal2", "TopVia2"}:
        add_layer(
            entry.get("Name"),
            (int(entry.get("Layer")), 0),
            float(entry.get("Zmin")),
            float(entry.get("Zmax")),
            entry.get("Material"),
            "via",
        )
add_layer("SUBGND", (250, 0), 0, 0, "LOWLOSS", "conductor")
oxide_top = 12.7303 if STACK_MODEL == "conformal" else 15.7303
passive_top = oxide_top + 0.4
add_layer("oxide_footprint", (902, 0), 0, oxide_top, "SiO2", "dielectric")
if STACK_MODEL == "conformal":
    add_layer("TM2_above", (900, 0), 14.2303, 16.1303, "SiO2", "dielectric")
    add_layer("TM2_sides", (901, 0), 11.2303, 14.2303, "SiO2", "dielectric")
stack = LayerStack(
    pdk_name=f"IHP SG13G2 / L6n2 {STACK_MODEL} benchmark",
    layers=layers,
    materials=materials,
    dielectrics=[
        {"name": "substrate", "zmin": -183.75, "zmax": -3.75, "material": "Substrate"},
        {"name": "EPI", "zmin": -3.75, "zmax": 0, "material": "EPI"},
        {"name": "SiO2", "zmin": 0, "zmax": oxide_top, "material": "SiO2"},
        {
            "name": "Passive",
            "zmin": oxide_top,
            "zmax": passive_top,
            "material": "Passive",
        },
        {
            "name": "AIR",
            "zmin": passive_top,
            "zmax": 16.1303 + AIR_MARGIN_UM,
            "material": "AIR",
        },
    ],
)
print(stack.validate_stack())
assert stack.validate_stack().valid
show_table(
    pd.DataFrame([dict(name=name, **layer.to_dict()) for name, layer in layers.items()])
)

## Configure and inspect the simulation size

Use absorbing outer boundaries, FEM order 2, a 0.1–14 GHz adaptive frequency sweep,
and no adaptive mesh refinement. The nominal conductor mesh sizes are not hard
bounds: thin oxide coverage and gaps create finer elements automatically.

The [upstream conformal, filled-metal study](https://github.com/VolkerMuehlhaus/gds2palace_ihp_sg13g2/blob/101e119fc08469fa3547eface93e1edc2dddfad7/more_examples/measured_vs_simulated/more_accurate_models_L6n2_v2/README.md)
reports the following order-2 sizes. gsim uses a different mesher configuration,
so read the actual notebook mesh statistics before allocating solver resources.

| Mesh | Tetrahedra | Published field DOFs | Published peak RAM |
|---|---:|---:|---:|
| 5 µm | 87,739 | 555,950 | 8.84 GB |
| 2 µm | 227,614 | 1,442,856 | 21.19 GB |
| 1 µm | 453,423 | 2,875,590 | 39.82 GB |

**Verified gsim preflight:** the default 2 µm conformal model produced **586,535 tets**
and **3,721,452 estimated field DOFs** locally (gsim 0.6.0, Gmsh 4.15.0, single-thread Delaunay). All signed element qualities were
positive; the worst tet-center distortion was κ≈4474. Some distorted elements remain.
This is a mesh/configuration check; no fresh Palace solve was performed for that check.
The live report below is authoritative when software versions or parameters change.

The estimated gsim field DOFs come from Nédélec mesh topology, before Palace's
preprocessing and boundary constraints. For order 2 this is `2 × edges + 2 × faces`.
These counts do not multiply by the number of output frequencies or port excitations.

In [ ]:
sim = DrivenSim()
sim.set_output_dir(run_dir)
sim.set_geometry(component)
sim.set_stack(stack)
sim.set_airbox(
    margin_x=AIR_MARGIN_UM,
    margin_y=AIR_MARGIN_UM,
    z_above=16.1303 + AIR_MARGIN_UM - passive_top,
    z_below=AIR_MARGIN_UM,
)
sim.set_numerical(
    order=FEM_ORDER, tolerance=1e-6, max_iterations=400, preconditioner="AMS"
)
sim.set_driven(
    fmin=FMIN_HZ,
    fmax=FMAX_HZ,
    num_points=NUM_POINTS,
    adaptive_tol=ADAPTIVE_TOL,
    adaptive_max_samples=30,
)
for name, _center, _width in port_definitions:
    sim.add_port(
        name,
        from_layer="SUBGND",
        to_layer="TopMetal1",
        geometry="interlayer",
        impedance=50,
        excited=True,
    )
validation = sim.validate_config()
print(validation)
assert validation.valid

In [ ]:
mesh_result = None
if BUILD_MESH:
    mesh_result = sim.mesh(
        preset="default",
        refined_mesh_size=MESH_SIZE_UM,
        max_mesh_size=100,
        merge_via_distance=0,
        planar_conductors=False,
        auto_size=False,
        verbose=False,
        algorithm_3d=MESH_ALGORITHM_3D,
        threads=1,
        surface_threads=1,
    )
    config_path = sim.write_config()
    print(mesh_result)
    actual_size = pd.DataFrame(
        [
            {
                "Model": STACK_MODEL,
                "Mesh (µm)": MESH_SIZE_UM,
                "Tetrahedra": mesh_result.mesh_stats["tetrahedra"],
                "Estimated field DOFs": mesh_result.mesh_stats["field_dofs"][
                    "estimated_field_dofs"
                ],
                "FEM order": FEM_ORDER,
            }
        ]
    )
    show_table(actual_size)
    actual_size.to_csv(run_dir / "mesh_size.csv", index=False)
    stack.to_yaml(run_dir / "benchmark_stack.yaml")
    config = json.loads(config_path.read_text())
    assert len(config["Boundaries"]["LumpedPort"]) == 2
    assert config["Boundaries"]["PEC"]["Attributes"]
    volume_tags = {
        group["name"]: group["tag"]
        for group in mesh_result.mesh_stats["groups"]["volumes"]
    }
    domain_materials = {
        tag: entry
        for entry in config["Domains"]["Materials"]
        for tag in entry["Attributes"]
    }
    for name in ["TopMetal1", "TopMetal2", "TopVia2", "Substrate", "EPI"]:
        assert name in volume_tags, f"Missing conductive/lossy domain: {name}"
        assert np.isclose(
            domain_materials[volume_tags[name]]["Conductivity"],
            materials[name]["conductivity"],
        ), f"Incorrect conductivity: {name}"
    if SHOW_MESH:
        sim.plot_mesh()
else:
    print("Mesh disabled. Set BUILD_MESH=True before running the solver.")

## Run a fresh gsim simulation

Set `RUN_SOLVER=True` in the parameter cell, then rerun from that cell onward.
`cloud` uses the same `sim.run(check_cache=True)` workflow as the other gsim Palace
notebooks. `local` uses Palace from `PALACE_EXECUTABLE`, `./bin/palace`, or PATH.
The run exports a 50 Ω Touchstone file in its model-specific output directory.
A solver failure is allowed to stop the notebook; an archived curve is never
substituted for a failed fresh simulation.

In [ ]:
simulated = None
if RUN_SOLVER:
    if mesh_result is None:
        raise RuntimeError("Build the mesh before running Palace.")
    if SOLVER_BACKEND == "cloud":
        sparams = sim.run(parent_dir=run_dir, check_cache=True, verbose="status")
    else:
        sparams = sim.run_local(num_processes=LOCAL_PROCESSES, verbose=True)
    names = ["P1", "P2"]
    s_matrix = np.empty((len(sparams.freq), 2, 2), dtype=complex)
    for output_index, output_name in enumerate(names):
        for input_index, input_name in enumerate(names):
            s_matrix[:, output_index, input_index] = sparams[
                (output_name, input_name)
            ].complex
    simulated = rf.Network(
        f=np.asarray(sparams.freq) * 1e9, s=s_matrix, z0=50, f_unit="hz"
    )
    simulated.name = f"gsim {STACK_MODEL}, {MESH_SIZE_UM:g} µm, order {FEM_ORDER}"
    simulated.write_touchstone(filename=str(run_dir / "gsim_l6n2"), form="ri")
    print(f"Fresh gsim result saved in {run_dir}")
else:
    print(
        "Solver not run. The comparisons below show measured and archived upstream data only."
    )

## Compare L, Q, resistance, and S-parameters

Q changes with conductor loss as well as mesh convergence. Compare 5 µm and 2 µm
gsim runs before a 1 µm check, using the same stack and port definition. Do not
tune conductivity to hide a mesh error. Negative inductance above SRF is expected.
The curves below include a fresh gsim trace only when the solver has completed.

The error table compares all four complex S-parameters against the measured data
on each candidate's frequency grid, with linear interpolation of measured real and
imaginary parts. RMS and maximum |ΔS| are dimensionless absolute differences;
they include phase and avoid relative errors at transmission zeros.

In [ ]:
networks = dict(reference_networks)
for result_file in SAVED_GSIM_RESULTS:
    result_path = Path(result_file).expanduser().resolve()
    networks[f"gsim saved: {result_path.parent.name}"] = rf.Network(str(result_path))
if simulated is not None:
    networks[simulated.name] = simulated
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
for name, network in networks.items():
    data = differential_data(network)
    style = (
        {"lw": 2.5, "color": "black"} if name.startswith("Measured") else {"lw": 1.4}
    )
    for ax, quantity in zip(axes, ["L_nH", "Q", "R_ohm"], strict=True):
        ax.plot(data.f_GHz, data[quantity], label=name, **style)
for ax, ylabel in zip(
    axes, ["Differential L (nH)", "Differential Q", "Differential R (Ω)"], strict=True
):
    ax.set(xlabel="Frequency (GHz)", ylabel=ylabel, xlim=(0.1, 14))
    ax.grid(alpha=0.3)
axes[0].set_ylim(-20, 35)
axes[1].set_ylim(-5, 20)
axes[2].set_yscale("log")
axes[1].legend(fontsize=7)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for name, network in networks.items():
    mask = (network.f >= FMIN_HZ) & (network.f <= FMAX_HZ)
    for ax, (row, column), label in zip(
        axes, [(0, 0), (1, 0)], ["S11", "S21"], strict=True
    ):
        db = 20 * np.log10(np.maximum(np.abs(network.s[mask, row, column]), 1e-15))
        ax.plot(network.f[mask] / 1e9, db, label=name)
        ax.set(xlabel="Frequency (GHz)", ylabel=f"{label} (dB)", xlim=(0.1, 14))
        ax.grid(alpha=0.3)
axes[1].legend(fontsize=7)
plt.show()

In [ ]:
comparison = pd.DataFrame(
    {name: metrics(network) for name, network in networks.items()}
).T
measured_metrics = comparison.loc["Measured (de-embedded)"]
for quantity in ["L @ 1 GHz (nH)", "Peak Q", "SRF (GHz)"]:
    comparison[f"{quantity}: error (%)"] = 100 * (
        comparison[quantity] / measured_metrics[quantity] - 1
    )
show_table(comparison.round(3))
s_errors = pd.DataFrame(
    {name: s_parameter_errors(network, measured) for name, network in networks.items()}
).T
show_table(s_errors.round(5))
run_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(run_dir / "comparison.csv")
s_errors.to_csv(run_dir / "s_parameter_errors.csv")
provenance = {
    "upstream_commit": upstream_commit,
    "source_checksums": checksums,
    "stack_model": STACK_MODEL,
    "mesh_size_um": MESH_SIZE_UM,
    "fem_order": FEM_ORDER,
    "via_fill_factor": via_fill_factor,
    "fresh_gsim_result": simulated is not None,
    "ground_model": "PEC sheet at z=0",
    "frequency_hz": [FMIN_HZ, FMAX_HZ],
    "gmsh_version": gmsh.__version__,
    "gsim_version": gsim.__version__,
    "gdsfactory_version": version("gdsfactory"),
    "ihp_gdsfactory_version": version("ihp-gdsfactory"),
    "mesh_algorithm_3d": MESH_ALGORITHM_3D,
}
(run_dir / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")

## Validation procedure

1. **Check the inputs and analysis first.** With `RUN_SOLVER=False`, verify the downloaded
   SHA-256 hashes and reproduce measured L≈5.009 nH at 1 GHz, peak Q≈15.957, and
   SRF≈11.070 GHz. The archived 1 µm conformal result gives peak Q≈14.864 and
   SRF≈11.030 GHz. Reproducing those values checks the data conversion.
2. **Check the mesh and configuration.** Confirm both 50 Ω ports, the PEC ground,
   the three filled conductive domains, the lossy substrate/EPI, and positive signed
   element quality. Record tets, estimated DOFs, distortion, and solver versions.
   A successful mesh check alone does not establish solver convergence.
3. **Run the gsim model.** Enable `RUN_SOLVER=True` with `STACK_MODEL="conformal"`.
   Run 5 µm for an initial comparison, then 2 µm and 1 µm with the same stack,
   FEM order, ports, and conductivities. Review Palace residuals and convergence
   messages as well as the returned S-parameters. Check sweep interpolation with
   a tighter `ADAPTIVE_TOL`, or `ADAPTIVE_TOL=0` for a full frequency sweep.
4. **Establish mesh convergence.** Add the saved `.s2p` files to `SAVED_GSIM_RESULTS`.
   Compare the changes in L at 1 GHz, peak Q, SRF, and complex S-parameters as the
   mesh is refined. Choose the required tolerances before reviewing results and
   report the observed changes; an upstream mesh count is not a convergence criterion.
5. **Compare with the experiment.** Review the L/Q/R and S11/S21 overlays, metric
   percentage errors, and the RMS/maximum complex |ΔS| table over 0.1–14 GHz.
   The Touchstone file is already de-embedded: apply an additional correction only
   if it is documented. Separate discretization changes from model differences,
   particularly the PEC ground sheet versus the upstream ground block and the
   via fill-factor approximation. Export `comparison.csv`, `s_parameter_errors.csv`,
   and `provenance.json` with the final Touchstone file.

Both gsim and the archived numerical reference use Palace. The independent comparison
is the supplied experimental WinCal Touchstone data. The present checked result is
mesh/configuration and analysis preflight; **fresh gsim solver convergence and agreement
with the measurement remain to be demonstrated**.

## Reading the result

The measurement has peak Q ≈15.96 and SRF ≈11.07 GHz. The archived finest conformal
result has peak Q ≈14.86 and SRF ≈11.03 GHz. These are useful checks on the data
conversion, not acceptance limits for a fresh gsim result.

A 2 µm to 1 µm comparison establishes discretization sensitivity for the selected
gsim model. Agreement with the measured curve also depends on ground geometry,
passivation, via homogenization, substrate properties, and measurement de-embedding.
This notebook does not infer fabrication provenance or claim a fresh simulation
is validated until its measured-data comparison has been reviewed.